# Rozdział 9 — Projektowanie kompletnego pipeline’u DSP

Wykonaj komórki po kolei z katalogu pakietu lub z `notebooks/`. Notebook
generuje siedem figur, tabele kontrolne i konfigurację z wyuczonym stanem.
Dane i kod są w pakiecie. Rozdział 9 zawiera formalizm i interpretację.
Notebook pracuje na części rozwojowej M8. Rekord 200 jest zarezerwowany
dla rozdziału 10.

Rewizja D01 (12 listopada 2026): położenia adnotacji EKG pochodzą z oryginalnych plików WFDB `.atr`, v1.0.0. Próbki sygnału pozostają zgodne z wycinkami CPSB. Ślad korekty: `chapter_notes/korekta_adnotacji_D01.md`.


In [1]:
from copy import deepcopy
import json
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import fftconvolve

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/ch09_candidate.json").is_file())
sys.path.insert(0, str(root / "src"))

from bspsbook.ch09 import (
    DETECTORS, LABELS, check_time_axis, detect, fit_train, load_development,
    quality_windows, save_design, score_development, write_csv,
)
from bspsbook.qrs_core import (
    evaluate_result, event_metrics, make_semisynthetic_variant,
)

plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.20, "legend.frameon": False,
    "svg.hashsalt": "CPSB-ch09",
})
colors = {"baseline": "#095b89", "matched_filter": "#a44b24"}
figures = root / "figures/generated"
figures.mkdir(parents=True, exist_ok=True)
candidate = json.loads((root / "config/ch09_candidate.json").read_text(encoding='utf-8'))
seed = candidate["random_seed"]
assert candidate["locked_test"]["run"] is False


def save(fig, slug):
    fig.savefig(figures / f"{slug}.png", dpi=175, metadata={"Software": "CPSB"})
    fig.savefig(figures / f"{slug}.svg", metadata={"Date": None})
    plt.close(fig)

## E9.1 — Przesunięcie zdarzeń a odstępy RR

In [2]:
fs = 360.0
ref = np.rint(np.arange(0.6, 4.61, 0.8) * fs).astype(int)
det = ref + 14
example, pair = event_metrics(ref, det, fs)
assert example["f1"] == 1 and len(pair.errors_samples) == 6
assert np.isclose(example["timing_mae_ms"], 14 / fs * 1000)
np.testing.assert_array_equal(np.diff(ref), np.diff(det))
assert event_metrics(np.array([100]), np.array([118]), fs)[0]["tp"] == 1
assert event_metrics(np.array([100]), np.array([119]), fs)[0]["f1"] == 0
assert event_metrics(np.array([100]), np.array([], dtype=int), fs)[0]["f1"] == 0
close, closest = event_metrics(np.array([100]), np.array([96, 102]), fs)
assert close["tp"] == 1 and close["fp"] == 1 and closest.errors_samples.tolist() == [2]

fig, ax = plt.subplots(figsize=(8.2, 2.6), layout="constrained")
for a, b in zip(ref / fs, det / fs):
    ax.plot([a, b], [1, 0], color="#89929b", lw=0.8)
ax.scatter(ref / fs, np.ones(len(ref)), marker="|", s=240, color="#333333")
ax.scatter(det / fs, np.zeros(len(det)), marker="|", s=240, color=colors["baseline"])
ax.set(yticks=[0, 1], yticklabels=["Detekcja", "Odniesienie"], ylim=(-.45, 1.45),
       xlabel="Czas [s]", title="Stałe przesunięcie 14 próbek, odstępy bez zmian")
save(fig, "ch09_01_zdarzenia")
print(f"F1 = {example['f1']:.3f}, MAE czasu = {example['timing_mae_ms']:.2f} ms, HR = {60 / np.median(np.diff(ref) / fs):.0f}/min")

F1 = 1.000, MAE czasu = 38.89 ms, HR = 75/min


## E9.2 — Audyt sześciu fragmentów

In [3]:
train, validation, audit = load_development(root)
normal, artifact = (next(f for f in validation if f["fragment_id"] == name)
                    for name in ("105_normal", "105_artifact"))
write_csv(root / "data/ch09_audit.csv", audit)
for row in audit:
    print(row["split"], row["fragment_id"], row["samples"], row["n_reference_beats"])
try:
    check_time_axis(normal["time_s"] + 1 / fs, fs, len(normal["time_s"]))
except ValueError:
    print("Odrzucono oś z poprawnym krokiem, lecz przesuniętym początkiem.")
else:
    raise AssertionError("Niewykryte przesunięcie osi")

fig, axes = plt.subplots(2, 1, figsize=(8.2, 4.5), sharex=True, sharey=True,
                         layout="constrained")
for ax, f, start in zip(axes, (normal, artifact), (29, 30)):
    t = f["time_s"] - start
    visible = (t >= 0) & (t <= 4)
    ax.plot(t[visible], f["signal_mV"][visible], color="#23374b", lw=0.9)
    for r in f["r_samples"][visible[f["r_samples"]]]:
        ax.axvline(t[r], lw=.7, color="#875b34", alpha=.65)
    ax.set(ylabel="EKG [mV]", title=f"{f['fragment_id']} · od {start} s czasu lokalnego")
axes[-1].set(xlabel="Czas od początku wycinka [s]", xlim=(0, 4))
save(fig, "ch09_02_wejscie")

train 100_normal 32400 117
train 100_ectopy 37800 129
train 103_normal 32400 106
train 103_apc 27000 87
validation 105_normal 32400 125
validation 105_artifact 32400 130
Odrzucono oś z poprawnym krokiem, lecz przesuniętym początkiem.


## E9.3 i E9.4 — Wariant energetyczny i wzorzec

In [4]:
state = fit_train(train, candidate)
meta = state["template_meta"]
h = state["template"]
anchor = meta["r_anchor_sample"]
center = (len(h) - 1) // 2
assert (len(h), anchor, center, meta["n_training_beats"]) == (181, 72, 90, 430)
assert np.isclose(np.linalg.norm(h), 1)
assert np.isclose(h.mean(), 0, atol=1e-12)
placed = np.zeros(512)
offset = 120
placed[offset:offset + len(h)] = h
corr = fftconvolve(placed, h[::-1], mode="same")
assert int(np.argmax(corr)) == offset + center
assert int(np.argmax(corr)) + anchor - center == offset + anchor

fig, ax = plt.subplots(figsize=(8.2, 3.1), layout="constrained")
ax.plot((np.arange(len(h)) - anchor) * 1000 / fs, h,
        color=colors["matched_filter"], label="Wzorzec po normalizacji")
ax.axvline(0, color="#333333", ls="--", label="Kotwica R")
ax.axvline((center - anchor) * 1000 / fs, color="#757c87", ls=":",
           label="Środek wektora wzorca")
ax.set(xlabel="Czas względem kotwicy [ms]", ylabel="Wartość wzorca [–]")
ax.legend(loc="upper right")
save(fig, "ch09_04_wzorzec")
print(state["baseline"], meta, state["matched"])

base_initial = detect(artifact, "baseline", candidate, state)
t = artifact["time_s"]
view = (t >= 29.5) & (t <= 35.5)
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8.2, 6.2), layout="constrained")
axes[0].plot(t[view], artifact["signal_mV"][view], color="#23374b", lw=.85)
axes[0].set(ylabel="EKG [mV]")
axes[1].plot(t[view], base_initial.filtered[view], color="#23374b", lw=.85)
axes[1].set(ylabel="Po filtracji [mV]")
axes[2].plot(t[view], base_initial.score[view], color=colors["baseline"], lw=1)
axes[2].axhline(6, color="#50555b", ls="--", label="Próg początkowy")
peak = base_initial.score_peak_samples[view[base_initial.score_peak_samples]]
axes[2].scatter(t[peak], base_initial.score[peak], s=20, color=colors["baseline"],
                label="Maksima wyniku")
for r in artifact["r_samples"][view[artifact["r_samples"]]]:
    axes[1].axvline(t[r], lw=.7, color="#875b34", alpha=.65)
rr = base_initial.r_samples[view[base_initial.r_samples]]
axes[1].scatter(t[rr], base_initial.filtered[rr], s=18, facecolors="none",
                edgecolors=colors["baseline"], label="Detekcje po doprecyzowaniu")
lo, hi = axes[1].get_ylim()
axes[1].set_ylim(lo, hi + 0.17 * (hi - lo))
axes[1].legend(loc="upper right")
axes[2].set(ylabel="Wynik energii [–]", xlabel="Czas lokalny [s]", xlim=(29.5, 35.5))
axes[2].legend(loc="upper right")
save(fig, "ch09_03_energia")

{'energy_center_train': 0.0019613346656797686, 'energy_scale_train': 0.0024361895034655514, 'energy_window_samples_by_fs': {'360': 43}, 'n_train_score_samples': 126720} {'fs_hz': 360.0, 'r_anchor_sample': 72, 'post_samples': 108, 'n_training_beats': 430, 'template_length_samples': 181} {'matched_local_scale_floor': 0.021196218659907617, 'matched_local_scale_train_median': 0.21196218659907617, 'local_background_window_samples': 721}


## E9.5 — Punkty pośrednie obu torów

In [5]:
match_initial = detect(artifact, "matched_filter", candidate, state)
assert match_initial.metadata["anchor_shift_samples"] == -18
np.testing.assert_array_equal(base_initial.evaluation_mask, match_initial.evaluation_mask)
np.testing.assert_array_equal(
    match_initial.candidate_r_samples, match_initial.score_peak_samples - 18,
)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.2, 5), layout="constrained")
for ax, method, result in zip(axes, DETECTORS, (base_initial, match_initial)):
    shift = result.metadata.get("anchor_shift_samples", 0) / fs
    ax.plot(t[view] + shift, result.score[view], lw=1, color=colors[method])
    ax.axhline(result.metadata["threshold_k_mad"], ls="--", lw=.8, color="#666666",
               label="Próg początkowy")
    m, pairs, references = evaluate_result(artifact, result, candidate)
    rs = references[(t[references] >= 29.5) & (t[references] <= 35.5)]
    ax.plot(t[rs], np.full(len(rs), .07), "|", color="#5b5351",
            transform=ax.get_xaxis_transform(), ms=9, label="Adnotacje")
    dr = result.r_samples[(t[result.r_samples] >= 29.5) & (t[result.r_samples] <= 35.5)]
    ax.plot(t[dr], np.full(len(dr), .16), "|", color=colors[method],
            transform=ax.get_xaxis_transform(), ms=9, label="Detekcje końcowe")
    ax.set(ylabel=f"Wynik · {LABELS[method]} [–]")
    ax.legend(loc="upper right", ncol=3, fontsize=8.5)
axes[-1].set(xlabel="Czas lokalny w układzie kotwicy [s]", xlim=(29.5, 35.5))
save(fig, "ch09_05_tory")

## E9.6 — Wybór progów i sprawdzenie polaryzacji

In [6]:
frozen, grid = score_development(validation, candidate, state)
assert len(grid) == 11
for x in validation:
    left = detect(x, "baseline", frozen, state)
    right = detect(x, "matched_filter", frozen, state)
    _, left_pairs, _ = evaluate_result(x, left, frozen)
    _, right_pairs, _ = evaluate_result(x, right, frozen)
    np.testing.assert_array_equal(left_pairs.reference_indices, right_pairs.reference_indices)
    np.testing.assert_array_equal(left.r_samples[left_pairs.detected_indices],
                                  right.r_samples[right_pairs.detected_indices])
assert {r["method"]: r["k"] for r in grid if r["selected"]} == {
    "baseline": 9.0, "matched_filter": 3.0,
}
write_csv(root / "data/ch09_threshold_grid.csv", grid)

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4), layout="constrained")
for ax, method in zip(axes, DETECTORS):
    rows = [r for r in grid if r["method"] == method]
    ax.plot([r["k"] for r in rows], [r["micro_f1"] for r in rows],
            "o-", color=colors[method], ms=4)
    best = next(r for r in rows if r["selected"])
    ax.scatter(best["k"], best["micro_f1"], marker="o", s=95,
               edgecolors=colors[method], facecolors="none", label="Wybór")
    ax.set(xlabel="Próg k [–]", title=LABELS[method].capitalize(),
           xticks=[r["k"] for r in rows])
    ax.set_ylim((.985, .997) if method == "baseline" else (.50, 1.035))
    ax.legend(loc="lower left")
axes[0].set_ylabel("F1 po zsumowaniu liczników [–]")
save(fig, "ch09_06_progi")

inverse = make_semisynthetic_variant(normal, "polarity", -1, seed, frozen)
metrics_rows = []
for f in (normal, artifact, inverse):
    outcomes = {m: detect(f, m, frozen, state) for m in DETECTORS}
    np.testing.assert_array_equal(outcomes["baseline"].evaluation_mask,
                                  outcomes["matched_filter"].evaluation_mask)
    for method, result in outcomes.items():
        m, _, _ = evaluate_result(f, result, frozen)
        row = {"fragment_id": f["fragment_id"], "method": method,
               "n_reference_all": len(f["r_samples"]), **m}
        metrics_rows.append(row)
        print(f["fragment_id"], method, m["tp"], m["fp"], m["fn"], round(m["timing_mae_ms"], 3))
write_csv(root / "data/ch09_development_results.csv", metrics_rows)
np.testing.assert_array_equal(detect(normal, "baseline", frozen, state).score,
                              detect(inverse, "baseline", frozen, state).score)
np.testing.assert_allclose(detect(normal, "matched_filter", frozen, state).score,
                           -detect(inverse, "matched_filter", frozen, state).score,
                           atol=1e-11)

105_normal baseline 122 0 0 0.319
105_normal matched_filter 122 0 0 0.319
105_artifact baseline 126 3 0 0.154
105_artifact matched_filter 126 0 0 0.154
105_normal__polarity_-1_s20260714 baseline 122 0 0 0.319
105_normal__polarity_-1_s20260714 matched_filter 120 0 2 0.324


## E9.7 — Brak danych i dostępność HR

In [7]:
gap = make_semisynthetic_variant(
    normal, "gap_s", frozen["design"]["gap_duration_s"], seed, frozen,
)
strict = deepcopy(frozen)
strict["quality"]["sqi_min"] = frozen["design"]["sqi_strict_diagnostic"]
quality_rows = []
for f in (normal, artifact, gap):
    for method in DETECTORS:
        result = detect(f, method, frozen, state)
        for policy, cfg in (("project", frozen), ("strict_diagnostic", strict)):
            rows = list(quality_windows(f, result, cfg))
            assert all((r["hr_bpm"] is not None) == r["result_available"] for r in rows)
            quality_rows.extend({"policy": policy, **r} for r in rows)
write_csv(root / "data/ch09_window_quality.csv", quality_rows)

def gap_rows(method):
    return [r for r in quality_rows if r["fragment_id"] == gap["fragment_id"]
            and r["policy"] == "project" and r["method"] == method]


for method in DETECTORS:
    failed = [r for r in gap_rows(method) if not r["result_available"]]
    assert len(failed) == 1
    r = failed[0]
    assert (r["start_s"], r["end_s"], r["missing_reason"], r["hr_bpm"]) == (
        40.0, 50.0, "incomplete_data", None,
    )
    assert r["valid_sample_fraction"] == 0.7 and r["evaluation_sample_fraction"] == 0.6
    assert r["sqi"] > frozen["quality"]["sqi_min"]
    print(method, f"SQI w oknie 40–50 s = {r['sqi']:.3f}, HR = brak")

availability = {}
for policy in ("project", "strict_diagnostic"):
    for method in DETECTORS:
        rows = [r for r in quality_rows if r["policy"] == policy and r["method"] == method
                and r["fragment_id"] in {"105_normal", "105_artifact"}]
        availability[policy, method] = (sum(r["result_available"] for r in rows), len(rows))
        print(policy, method, availability[policy, method])

fig, axes = plt.subplots(3, 1, figsize=(8.2, 6), layout="constrained")
x = np.where(gap["valid_mask"], gap["signal_mV"], np.nan)
axes[0].plot(gap["time_s"], x, lw=.8, color="#23374b")
axes[0].axvspan(40, 44, color="#d79161", alpha=.18, label="Poza maską oceny")
axes[0].axvspan(40.5, 43.5, color="#d79161", alpha=.35, label="Brak danych")
axes[0].set(xlim=(36, 48), ylabel="EKG [mV]", xlabel="Czas lokalny [s]")
axes[0].legend(loc="lower right", ncol=2, fontsize=8.5)
for method in DETECTORS:
    rows = gap_rows(method)
    centers = [(r["start_s"] + r["end_s"]) / 2 for r in rows]
    axes[1].plot(centers, [r["sqi"] for r in rows], "o-", ms=4,
                 color=colors[method], label=LABELS[method])
    axes[2].plot(centers, [r["hr_bpm"] for r in rows], marker="o", ms=4,
                 ls="-" if method == "baseline" else "--", color=colors[method])
axes[1].axhline(.55, ls="--", lw=.8, color="#666666", label="Próg projektu")
axes[1].set(ylabel="SQI [–]", ylim=(.45, 1))
axes[1].legend(loc="lower right", ncol=3, fontsize=8.5)
axes[2].set(ylabel="HR [1/min]", xlabel="Środek okna [s]")
for ax in axes[1:]:
    ax.axvspan(40, 50, color="#d79161", alpha=.18)
    ax.set_xlim(0, 90)
save(fig, "ch09_07_dostepnosc")

baseline SQI w oknie 40–50 s = 0.723, HR = brak
matched_filter SQI w oknie 40–50 s = 0.829, HR = brak
project baseline (18, 18)
project matched_filter (18, 18)
strict_diagnostic baseline (0, 18)
strict_diagnostic matched_filter (15, 18)


## E9.8 — Przekazanie konfiguracji do rozdziału 10

In [8]:
handoff = save_design(root, frozen, state)
report = {
    "handoff": handoff,
    "selected": [r for r in grid if r["selected"]],
    "real_window_availability": {
        f"{policy}/{method}": counts for (policy, method), counts in availability.items()
    },
    "test_executed": False,
}
(root / "data/ch09_run.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8",
newline='\n')
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "handoff": {
    "config_sha256_canonical": "8cfcc9df0d8d07fb9506606737f35bda0d36f55ad78e7b86306ff5b905302952",
    "state_file_sha256": "bbbc5248a5cf4522c8a29c507da86916813e38ea51c3d5b186c6522dc8f488a5",
    "manifest_sha256": "47cf744b4a72d1e1cb11ca72369f50b56f5f69a523ce80ad2d4f3b4799e81810",
    "candidate_file_sha256": "86007b4b84e7e520eaa6e443e69fecb3271ab299d5323bfd4ab693291156435d",
    "core_sha256": "ca2d633a5b52685c3276837b2811033f1a31c38464126527ead2b41962ec56a6",
    "orchestration_sha256": "d33fce4ccbe9a62a0471d406d8c8c288b98ee7cfbdca3b2f3d6b9dc7dde74eb2",
    "fit_records": [
      "100",
      "103"
    ],
    "selection_records": [
      "105"
    ],
    "reserved_test_record": "200"
  },
  "selected": [
    {
      "method": "baseline",
      "k": 9.0,
      "tp": 248,
      "fp": 3,
      "fn": 0,
      "micro_f1": 0.9939879759519038,
      "timing_mae_ms": 0.23521505376344085,
      "selected": true
    },
    {
      "method": "matched_filter",
      "k": 3.0,
 